<div align="left">
    <img src="https://form.ethz.ch/hss26/_jcr_content/par/image/image.imageformat.1286.dpr2.1527502072.png" alt="Earth Observation Foundation Models Workspace" width="70%" style="max-height: 350px; object-fit: cover; border-radius: 8px; box-shadow: 0 4px 10px rgba(0,0,0,0.15);">
</div>

# Hands-on Session: Earth Observation (EO) Foundation Models

🌍 **Can an AI model read the Earth's surface — without ever being told what to look for?**
In this hands-on session, we explore geospatial foundation models and put them to work on real satellite imagery — extracting deep structural and spectral fingerprints of the landscape across time, reducing millions of features into interpretable patterns, and letting the model reveal what changed, when, and where. No labels. No manual rules. **Just intelligence learned from the Earth itself.**

### 🔍 Technical Workflow: Framework Execution

**Case study:** Trentino region — Val di Fiemme forest, devastated by the **Vaia storm (October 2018)**.  
We use this event as a ground-truth anchor to validate whether the pipeline can autonomously detect the abrupt landscape change without any labelled data.

**Pipeline steps:**

1. **Load & process HLS data from GEE**
   - Query the Harmonized Landsat Sentinel-2 (HLS) collection from Google Earth Engine
   - Filter by area of interest (Val di Fiemme) and time range spanning pre- and post-Vaia (2017–2019)

2. **Load the foundation models**
   - Initialize pre-trained weights of **Prithvi EO v2** via the `terratorch` registry (`prithvi_eo_v2_300`)
   - Initialize pre-trained weights of **Clay v1.5** via the `terratorch` registry (`timm_clay_v1_base`)

3. **Generate high-dimensional embeddings**
   - Pass image tensors through the Vision Transformer (ViT) encoder
   - Each observation date is mapped to a dense feature vector — a mathematical "fingerprint" of the landscape at that moment

4. **Feature reduction & visualization (PCA & t-SNE)**
   - **PCA** isolates the directions of maximum variance across dates, surfacing the most structurally significant changes
   - **t-SNE** projects the high-dimensional space into 2D, making temporal clusters and outliers (e.g. the post-Vaia observations) visually separable

5. **Change date detection**
   - Compare feature spaces across time steps
   - Identify the date where the embedding distribution shifts most abruptly — expected to align with **October 2018**

---

### 💡 Core Takeaways

#### 1. HLS data
- A radiometrically harmonized, analysis-ready product fusing Landsat and Sentinel-2
- Provides consistent multi-spectral time series — essential for temporal change analysis

#### 2. Geospatial foundation models
- **Multi-spectral design:** ingest any band combination (RGB, NIR, SWIR) in any order — not limited to 3-channel RGB
- **Self-supervised pre-training:** trained on millions of unlabelled global images via Masked Autoencoders (MAE), learning Earth surface physics with no manual annotation
- **Zero-shot capability:** no fine-tuning needed for downstream tasks like change detection

#### 3. Embeddings as landscape fingerprints
- Instead of pixel-by-pixel band comparisons, the model encodes the full structural pattern, canopy texture, and moisture status into a single vector
- Changes in the embedding space directly reflect real-world landscape changes — in our case, the mass windthrow caused by Vaia across Val di Fiemme.


## Requirements
- **Google Earth Engine** — free account + a GEE Cloud Project ID
- **Google Drive** — for exporting image chips (~100 MB)
- **Hugging Face** — free account (if needed)
- **GPU runtime** — Google Colab (T4) or local CUDA GPU (≥8 GB VRAM)




## Setup
Install required packages

In [ ]:
# Install packages
%pip install --upgrade -q pip
%pip install "numpy<2.1" geemap opencv-python "terratorch>=1.1.0" einops "scikit-learn>=1.5.0"

In [ ]:
#Load packages
import ee
import geemap

import cv2
import torch
import numpy as np
import time
from einops import rearrange, reduce
from matplotlib import pyplot as plt
from sklearn import decomposition, preprocessing
from terratorch import BACKBONE_REGISTRY
from sklearn.manifold import TSNE
from sklearn.impute import SimpleImputer
import math


### Step 1: Initialize GEE
We need `geemap` for interactive map visualizations natively within the Colab environment, and the `ee` library to communicate with the Google Earth Engine API.

### Prerequisites
Make sure you have access to a **Google Earth Engine (GEE)** account. When running the initialization cell, you will be prompted to authenticate through your browser.

### Google Earth Engine requires a Google Cloud Project ID to run.
#
### How to find your Project ID:
1. Open a new tab and go to: https://code.earthengine.google.com/
2. Look at the top-right corner next to your profile picture/name.
3. Copy the project text ID shown there (e.g., 'ee-yourusername' or 'my-project-1234').
4. Paste that text string inside the single quotes for PROJECT_ID below.

In [ ]:
# 1. Authenticate to Earth Engine
ee.Authenticate()

# 2. Define your Earth Engine Cloud Project ID here
# Students can find this on their Earth Engine code editor page or GCP console
PROJECT_ID = 'lively-crane-484414-n0'

# 3. Initialize with the project ID
ee.Initialize(project=PROJECT_ID)

### Step 2: Define Region of Interest (ROI)
We define a point using a specific latitude and longitude, create a buffer of 3,840 meters around it, and find its bounding box coordinates (`bounds()`) to use as our target window.

In [ ]:
lat = 46.231128
lon = 11.407454

ROI = ee.Geometry.Point([lon, lat]).buffer(3840).bounds()

print("ROI defined successfully.")

### Step 3: Load and Filter Base Image Collection
We target the **NASA/HLS/HLSS30/v002** image collection (Harmonized Landsat Sentinel-2). To ensure we don't handle empty images or data fragments, we write a mapping function to count valid pixels inside our ROI using `reduceRegion` and filter out any completely blank scenes.

In [ ]:
# Load the raw collection overlapping our ROI
collection = ee.ImageCollection('NASA/HLS/HLSS30/v002').filterBounds(ROI)

# Define helper mapping function to get valid pixel counts inside our ROI boundary
def count_pixels(img):
    count = img.reduceRegion(
        reducer=ee.Reducer.count(),
        geometry=ROI,
        scale=30,
        maxPixels=1e13
    ).values().get(0)
    return img.set('pixel_count', count)

# Map the function and filter collection
validCollection = collection.map(count_pixels).filter(ee.Filter.gt('pixel_count', 0))

print("Collection filtered for valid pixels successfully.")

### Step 4: Image Selection Strategy (2018 vs 2019)
We select imagery from two fixed yearly windows: **2018** and **2019**. We filter out heavily cloudy scenes (keeping less than 30% cloud cover), sort by clarity, and want to select the **30 images from each year**.

In [ ]:
# Filter 20 clearest images from 2018
Pool_2018 = (validCollection
              .filterDate('2018-01-01', '2018-12-31')
              .filter(ee.Filter.lt('CLOUD_COVERAGE', 30))
              .sort('CLOUD_COVERAGE'))

# Filter 20 clearest images from 2019
Pool_2019 = (validCollection
             .filterDate('2019-01-01', '2019-12-31')
             .filter(ee.Filter.lt('CLOUD_COVERAGE', 30))
             .sort('CLOUD_COVERAGE'))

# Select top 20 clearest from each year and merge
year_2018 = Pool_2018.limit(30)
year_2019 = Pool_2019.limit(30)
selected = year_2018.merge(year_2019)

print(f"2018 images:  {year_2018.size().getInfo()}")
print(f"2019 images: {year_2019.size().getInfo()}")
print(f"Total combined images:   {selected.size().getInfo()}")

### Step 5: Extracting Metadata & Dates
To make working with the images easier, we'll extract the system timestamp and parse it into a clean readable string format (`YYYY-MM-dd`). Then we will pull this date array into python's client side so you can easily choose which one to preview.

In [ ]:
# Map function to add readable text dates as metadata attribute
def add_date_str(img):
    date = ee.Date(img.get('system:time_start')).format('YYYY-MM-dd')
    return img.set('date_str', date)

withDate = selected.map(add_date_str)

# Convert Server-side Collection to Server List and fetch array attributes to client
list_images = withDate.toList(40)
dates = withDate.aggregate_array('date_str').getInfo()

print("Available Scene Dates mapped to indices:")
for idx, date_str in enumerate(dates):
    print(f"Index [{idx}]: {date_str}")

### STEP 6:  SELECTION (6 images from 2018 & 6 images from 2019)

This section filters the `HLS` image collection to identify and select the clearest, least cloudy images within our Region of Interest (ROI) for **2018** and **2019**:

1. **Unpack Bitmasks:** Decodes the `Fmask` band to flag clouds, shadows, adjacent cloud pixels, and snow/ice.
2. **Calculate Clear Ratio:** Computes the percentage of clear pixels relative to the total ROI area for each image.
3. **Select Best Candidates:** Sorts the collection by highest clear pixel ratio and retains the **top 6 clearest images** for each year (12 images total).

In [ ]:
def add_quality_metrics(img):
    fmask = img.select('Fmask')

    # Correct HLS bit positions (Bit 0 to Bit 4)
    cirrus   = fmask.bitwiseAnd(1 << 0).neq(0)
    cloud    = fmask.bitwiseAnd(1 << 1).neq(0)
    shadow   = fmask.bitwiseAnd(1 << 2).neq(0)
    adjacent = fmask.bitwiseAnd(1 << 3).neq(0)
    snow     = fmask.bitwiseAnd(1 << 4).neq(0)

    # Combined absolute clear condition mask
    clear = cirrus.Not().And(cloud.Not()).And(shadow.Not()).And(adjacent.Not()).And(snow.Not())

    valid_dict = clear.reduceRegion(
        reducer=ee.Reducer.sum(),
        geometry=ROI,
        scale=30,
        maxPixels=1e13
    )
    valid_count = valid_dict.get(valid_dict.keys().get(0))

    total_dict = fmask.reduceRegion(
        reducer=ee.Reducer.count(),
        geometry=ROI,
        scale=30,
        maxPixels=1e13
    )
    total_count = total_dict.get(total_dict.keys().get(0))

    clear_ratio = ee.Number(valid_count).divide(ee.Number(total_count))

    return img.set({
        'valid_count': valid_count,
        'total_count': total_count,
        'clear_ratio': clear_ratio
    })

# Form clear candidate tracking arrays straight from your working withDate collection
candidate_list = withDate.toList(withDate.size())
candidate_with_index = ee.ImageCollection(
    ee.List.sequence(0, withDate.size().subtract(1)).map(
        lambda i: ee.Image(candidate_list.get(i)).set('candidate_index', i)
    )
)
qualityCollection = candidate_with_index.map(add_quality_metrics)

# CRITICAL SAFETY EXTRA: Filter out any images that somehow returned null ratios
valid_quality_collection = qualityCollection.filter(ee.Filter.notNull(['clear_ratio']))

# Select the top 6 clearest images per year
best_2018 = valid_quality_collection.filterDate('2018-01-01', '2018-12-31').sort('clear_ratio', False).limit(6)
best_2019 = valid_quality_collection.filterDate('2019-01-01', '2019-12-31').sort('clear_ratio', False).limit(6)
best_quality_candidates = best_2018.merge(best_2019)

# Sync metadata values down to client memory
best_dates = best_quality_candidates.aggregate_array('system:time_start').map(
    lambda t: ee.Date(t).format('YYYY-MM-dd')
).getInfo()
clear_ratios = best_quality_candidates.aggregate_array('clear_ratio').getInfo()
candidate_indices = best_quality_candidates.aggregate_array('candidate_index').getInfo()

print("--- Automated Quality Metric Summary Check ---")
for idx, d, r in zip(candidate_indices, best_dates, clear_ratios):
    print(f"Candidate Index [{idx}]: {d}, clear_ratio = {r:.3f}")

# Extract ordered imagery using the metric verification sequence
list_images = best_quality_candidates.sort('system:time_start').toList(12)
sorted_dates = best_quality_candidates.sort('system:time_start').aggregate_array('date_str').getInfo()

###Step 6a: Visual Inspection
Before passing images into the model, we can visually inspect each scene.


In [ ]:
# CHANGE THIS INDEX VALUE TO TEST DIFFERENT TIMESTAMPS
viewIndex = 0

# Fetch the specified target image
imageToVisualize = ee.Image(list_images.get(viewIndex)).clip(ROI)
imageDateStr = dates[viewIndex]

# Build interactive map workspace
Map = geemap.Map()
Map.centerObject(ROI, 12)

# Configure True Color RGB Render parameter values
vis_params = {
    'bands': ['B4', 'B3', 'B2'],
    'min': 0.005,
    'max': 0.2,
    'gamma': 1.9
}

Map.addLayer(imageToVisualize, vis_params, f'HLS RGB Scene [{viewIndex}]: {imageDateStr}')
print(f"Displaying Scene captured on: {imageDateStr}")
Map

### Step 7: Batch Export to Google Drive

Before running the foundation model locally, we export the selected scenes from GEE to Google Drive as analysis-ready image chips.

**What gets exported:**
- **Bands:** `B2, B3, B4, B8A, B11, B12` — the exact 6-band (Blue, Green, Red, Narrow NIR, SWIR-1, SWIR-2)
- **Size:** 256×256 pixels per chip — the native patch size of the ViT encoder
- **Projection:** EPSG:32632 (UTM Zone 32N — appropriate for the Val di Fiemme area)
- **Format:** Float32, one file per date

One export task is submitted per image (12 total). Tasks run asynchronously in the GEE backend — monitor progress in the **GEE Task Manager** before moving to the next step.

In [ ]:
print("\nDownloading and processing HLS image arrays locally...")
image_arrays = []
for i in range(12):
    img_ee = ee.Image(list_images.get(i))
    mlInput = img_ee.select(['B2', 'B3', 'B4', 'B8A', 'B11', 'B12']).toFloat()
    img_np = geemap.ee_to_numpy(mlInput, region=ROI, scale=30)


    # Kill the NaNs before they contaminate cv2.resize

    if np.isnan(img_np).any():
        nan_pixel_count = np.isnan(img_np).sum()
        print(f" Warning: Image index [{i}] contains {nan_pixel_count} missing boundary/clip pixels. Patching...")
        # Replace NaNs with a neutral 0.0 value
        img_np = np.nan_to_num(img_np, nan=0.0, posinf=0.0, neginf=0.0)

    # Enforce precise spatial dimensions (256x256 pixels) for Prithvi specifications
    resized_bands = []
    for c in range(img_np.shape[2]):
        # This will now process perfectly with no NaN propagation
        band_resized = cv2.resize(img_np[:, :, c], (256, 256), interpolation=cv2.INTER_LINEAR)
        resized_bands.append(band_resized)
    img_np = np.stack(resized_bands, axis=-1)
    image_arrays.append(img_np)

full_stack = np.stack(image_arrays, axis=0) # Shape: (12, 256, 256, 6)
# Rearrange dimensions to channels first configuration layout: (12, 6, 256, 256)
full_stack = rearrange(full_stack, 'b h w c -> b c h w')

print(f"\n[SUCCESS]: Array successfully created in RAM with zero NaNs!")
print(f"Tensor Shape: {full_stack.shape} -> (Images: {full_stack.shape[0]}, Bands: {full_stack.shape[1]}, Height: {full_stack.shape[2]}, Width: {full_stack.shape[3]})")

### Step 7a: Visualization
Check the images all together.

In [ ]:
n = full_stack.shape[0]
cols = 4
rows = math.ceil(n / cols)
fig, axes = plt.subplots(rows, cols, figsize=(cols*3, rows*3))
axes = axes.flatten()

for i in range(n):
    img = full_stack[i]                       # (6, H, W)
    rgb = img[[2,1,0]].transpose(1,2,0)       # R=B4,G=B3,B=B2 -> (H,W,3)
    if rgb.max() > 1.0:                        # scale if raw reflectance
        rgb = rgb * 0.0001
    rgb = np.clip((rgb - 0.005) / (0.2 - 0.005), 0, 1)  # match your vis min/max
    rgb = rgb ** (1/1.9)                       # gamma 1.9
    axes[i].imshow(rgb)
    axes[i].set_title(sorted_dates[i], fontsize=9)
    axes[i].axis('off')

for j in range(n, len(axes)):
    axes[j].axis('off')
plt.tight_layout(); plt.show()

# Section 2: Generating Deep Geospatial Embeddings with Prithvi

Now that we have selected and verified our curated sequence of clean HLS images, we will leverage **Prithvi** a geospatial foundation model built by NASA and IBM.

Instead of treating pixels as isolated spectral numbers, Prithvi uses a 3D Vision Transformer (ViT) architecture to look at patches of your imagery across space and time simultaneously. It outputs dense multi-dimensional vector representations called **embeddings** that summarize semantic surface features (e.g., changes in vegetation, soil moisture, and land surface features caused by the storm).

### Step 8: Initialize Modeling Runtime Environment
We check for hardware device acceleration (CUDA GPU) and ensure the necessary geospatial model modules are ready for execution.

We installed the `terratorch` library framework along with `einops` for tensor reshaping operations, and `scikit-learn` to handle downstream evaluation tasks.

In [ ]:
# Verify if GPU acceleration is active
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using hardware device acceleration: {device}")
print(f"Active CV2 Version: {cv2.__version__}")

### Step 9: Instantiate and Load Pretrained Prithvi Model Weights
We pull the target backbone model (`prithvi_eo_v2_300`) directly from the registry, set it to evaluation mode, and move it to our active runtime hardware processing device.

In [ ]:
# Verify that our targeted backbone exists in the terratorch framework registry
model_name_target = "prithvi_eo_v2_300"
assert any(model_name_target in m for m in BACKBONE_REGISTRY), "Target model backbone not found!"

print("--> Downloading and instantiating Prithvi model weights from Hugging Face (~1.3 GB)...")
# Build the model using the registry with pretrained weights
model = BACKBONE_REGISTRY.build(model_name_target, pretrained=True)
model.to(device)
model.eval() # Set model to evaluation mode

print(f"\n[SUCCESS]: {model_name_target} successfully loaded and allocated to memory.")

### Step 10: Format & Shape Selected Imagery to Sequence Chips

Prithvi EO v2 expects inputs structured as tensor sequences: `[Batch, Channels, Time, Height, Width]`.

We prepare the data using **Sliding window chipping** — rather than fixed non-overlapping blocks, a window of **4 timesteps** slides one step at a time across the 12 scenes, producing **9 overlapping chips**. Each chip captures a short temporal sequence and consecutive chips share 3 images, giving the model finer temporal resolution to detect gradual or abrupt changes:

| Chip | Indices | Description |
|------|---------|-------------|
| 0 | 0 → 3 | Early 2018 |
| 1 | 1 → 4 | Mid 2018 |
| ... | ... | ... |
| 5 | 5 → 8 | Transition (pre → post Vaia) |
| ... | ... | ... |
| 8 | 8 → 11 | 2019 post-storm |

The chip covering the **October 2018** boundary is where we expect the largest shift in the embedding space.

In [ ]:
window_size = 4
chips = []

print("====== RUNNING DATA INTEGRITY SLIDING WINDOW BLOCK ======")

# Loop to create overlapping window chips
for start_idx in range(len(full_stack) - window_size + 1):
    end_idx = start_idx + window_size
    chip_slice = full_stack[start_idx:end_idx]  # Slices a (4, 6, 256, 256) segment


    # Clear hidden NaNs inside RAM
    if np.isnan(chip_slice).any() or np.isinf(chip_slice).any():
        nan_count = np.isnan(chip_slice).sum()
        inf_count = np.isinf(chip_slice).sum()
        print(f"Found pre-existing invalid numbers in Window Index [{start_idx}]! NaNs: {nan_count}, Infs: {inf_count}. Patching...")

        # Neutralize bad values directly in memory before they hit Prithvi
        chip_slice = np.nan_to_num(chip_slice, nan=0.0, posinf=0.0, neginf=0.0)

    # Add an explicit scaling check to verify values are within reasonable reflectance limits (0.0 to 1.0)
    # If your full_stack wasn't scaled by 0.0001 yet, do it here:
    if chip_slice.max() > 1.0:
        chip_slice = chip_slice * 0.0001

    # Keep layout as (t c h w) - our inference block will handle the rest
    chips.append(chip_slice)

print(f"\n[SUCCESS]: Formed {len(chips)} perfectly clean sliding window arrays.")
print(f"Individual target array shape: {chips[0].shape}\n")

### Step 11: Pass Chips Through Prithvi to Extract Embeddings
We execute the forward processing pipeline on the GPU workspace, extract the generalized spatial representation `[CLS]` token for each timeline sequence block, and construct our multi-dimensional embeddings map matrix.

In [ ]:
embeddings_list = []
start_time = time.time()

print("Executing forward pass inference with corrected Prithvi layout mapping...")
model.eval()

for idx, chip in enumerate(chips):
    if isinstance(chip, torch.Tensor):
        chip_tensor = rearrange(chip, "t c h w -> 1 c t h w").float().to(device)
    else:
        chip_tensor = rearrange(chip, "t c h w -> 1 c t h w")
        chip_tensor = torch.from_numpy(chip_tensor).float().to(device)

    with torch.no_grad():
        outputs = model(chip_tensor)

        if isinstance(outputs, dict):
            output_embedding = outputs.get("spatial_features", outputs.get("embeddings"))
        else:
            output_embedding = outputs

        cls_token = output_embedding[0][:, 0, :].detach().cpu().numpy().ravel()

        nan_in_token = np.isnan(cls_token).sum()
        inf_in_token = np.isinf(cls_token).sum()

        if nan_in_token > 0 or inf_in_token > 0:
            print(f"⚠️ [ALERT] Chip [{idx}] generated invalid values! NaNs: {nan_in_token}, Infs: {inf_in_token}")
        else:
            print(f"-> Generated embedding vector for Chip [{idx}] with dimensions: {cls_token.shape} (All values clean)")

        embeddings_list.append(cls_token)

embeddings_matrix = np.array(embeddings_list)
print(f"\n[SUCCESS]: Processed all matrices in {time.time() - start_time:.2f} seconds.")


### Step 12: Embedding Extraction & PCA Trajectory Analysis

With the 9 sliding window chips ready, we pass each through Prithvi EO v2 and analyse how the embedding space evolves over time.

**1. Embedding extraction**
Each chip of shape `(1, 6, 4, 256, 256)` is fed into the model. The output feature tensor is flattened into a single vector per window, producing a **(9, N)** embedding matrix — one row per temporal window.

**2. Normalization & PCA**
The matrix is standardized with `StandardScaler` and projected onto the **top 2 principal components**:
- **PC1** captures seasonal effects, moisture, or partial disturbance
- **PC2** captures the dominant axis of landscape variance — typically driven by large structural changes such as canopy loss

**3. Trajectory visualization**
The two panels plot each window's PCA coordinate over time (W0 → W8), colored blue-to-red chronologically:
- A **smooth trajectory** indicates gradual seasonal change
- An **abrupt jump** between consecutive windows signals a sudden landscape shift — expected around the window spanning **October 2018**

> 💡 *Look for the window where PC1 or PC2 shows the largest single step — that window contains the Vaia storm transition.*

In [ ]:
print("1. Extracting high-dimensional embeddings from the 9 true sliding window chips...")
rolling_embeddings = []

# chips contains 9 segments of shape (4, 6, 256, 256) from your working Step 10
for i in range(len(chips)):
    chip_sequence = chips[i]  # Shape: (Time=4, Bands=6, H=256, W=256)

    # Rearrange layout to match Prithvi: [Channels, Time, Height, Width]
    chip_transposed = np.transpose(chip_sequence, (1, 0, 2, 3)) # Becomes (6, 4, 256, 256)

    # Add batch dimension -> Shape becomes (1, 6, 4, 256, 256)
    input_tensor = torch.from_numpy(chip_transposed).float().unsqueeze(0).to(device)

    with torch.no_grad():
        try:
            features = model(input_tensor)
        except NameError:
            features = backbone(input_tensor)

        # Unpack output layers safely
        if isinstance(features, list):
            feature_tensor = features[-1]
        elif isinstance(features, dict):
            first_key = list(features.keys())[0]
            feature_tensor = features[first_key]
        else:
            feature_tensor = features

        # Extract to CPU and flatten completely
        feature_np = feature_tensor.detach().cpu().numpy()
        flat_vector = feature_np.flatten()
        rolling_embeddings.append(flat_vector)

# Create the 2D matrix safely in local memory
rolling_matrix = np.vstack(rolling_embeddings)
print(f"[SUCCESS]: Generated matrix of shape: {rolling_matrix.shape}")


# 2. STANDARD SCALING & MULTI-COMPONENT PCA

print("\n2. Normalizing feature variances...")
scaler = preprocessing.StandardScaler()
scaled_matrix = scaler.fit_transform(rolling_matrix)

print("3. Extracting top 2 principal components to decouple environmental factors...")
pca_rolling = decomposition.PCA(n_components=2)
pca_results = pca_rolling.fit_transform(scaled_matrix)

var1 = pca_rolling.explained_variance_ratio_[0] * 100
var2 = pca_rolling.explained_variance_ratio_[1] * 100
print(f"   └─► Component 1 accounts for: {var1:.1f}% variance")
print(f"   └─► Component 2 accounts for: {var2:.1f}% variance")


# 3. SIDE-BY-SIDE TRAJECTORY VISUALIZATION

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 6))
window_colors = plt.cm.coolwarm(np.linspace(0, 1, len(chips)))
window_ticks = [f"W{i}" for i in range(len(chips))]

# --- Plot PCA Component 1 ---
for i in range(len(chips)):
    ax1.scatter(i, pca_results[i, 0], color=window_colors[i], s=200, edgecolors='black', zorder=3)
ax1.plot(range(len(chips)), pca_results[:, 0], color='black', linestyle='--', alpha=0.5, zorder=2)
ax1.axhline(0, color='gray', linestyle='--', linewidth=0.8)
ax1.set_xticks(range(len(chips)), window_ticks)
ax1.set_xlabel("Sliding Window Chronological Steps")
ax1.set_ylabel("PCA Component 1 Coordinate")
ax1.set_title(f"PCA Component 1 Variance Path ({var1:.1f}%)")
ax1.grid(True, alpha=0.2)

# --- Plot PCA Component 2 ---
for i in range(len(chips)):
    label_text = f"W{i}: {sorted_dates[i]} to {sorted_dates[i+3]}"
    ax2.scatter(i, pca_results[i, 1], color=window_colors[i], s=200, edgecolors='black', label=label_text, zorder=3)
ax2.plot(range(len(chips)), pca_results[:, 1], color='black', linestyle='--', alpha=0.5, zorder=2)
ax2.axhline(0, color='gray', linestyle='--', linewidth=0.8)
ax2.set_xticks(range(len(chips)), window_ticks)
ax2.set_xlabel("Sliding Window Chronological Steps")
ax2.set_ylabel("PCA Component 2 Coordinate")
ax2.set_title(f"PCA Component 2 Variance Path ({var2:.1f}%)")
ax2.grid(True, alpha=0.2)

# Place legend cleanly outside the subplots
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', title="Window Span Dates")
plt.tight_layout()
plt.show()


### Step 13: Standalone Date Evaluation & Temporal Workaround

**Objective:** Analyze all 12 satellite captures individually to map day-by-day landscape evolution.

#### 1. The 4x Temporal Workaround
* **Problem:** Prithvi's 3D ViT architecture expects `Time=4`. A single date (`Time=1`) causes a dimension mismatch.
* **Fix:** Replicate the single date 4 times using `np.expand_dims` and `np.repeat` to create a valid `[1, 6, 4, 256, 256]` tensor.

#### 2. High-Dimensional PCA Analysis
* **Extraction:** Pulls the raw, un-averaged feature block (**1,049,600 features/day**) to preserve local details.


### Step 13 (Extended): Non-Linear Spatial Clustering via t-SNE

#### **Why Add t-SNE to Individual Dates?**
While PCA excels at identifying global linear variance (like tracking continuous timeline shifts or separating cloud noise), it can struggle to capture localized, complex multi-dimensional geometric groupings.

To provide an alternative analytical viewpoint, we implement **t-Distributed Stochastic Neighbor Embedding (t-SNE)**—a non-linear dimensionality reduction technique explicitly optimized to map high-dimensional structural proximities into distinct low-dimensional neighborhood clusters.

#### **Key Implementation Details**
* **Safe Perplexity Calibration:** Because our dataset is limited to 12 standalone dates, we explicitly constrain the hyperparameter (`perplexity=3`). This ensures local probabilistic neighborhoods are calculated accurately without breaking on a small sample size.
* **Comparing Multi-Manifold Projections:** By feeding t-SNE the exact same standardized feature matrix used for PCA, students can directly contrast how **linear trajectory tracking** (PCA) vs. **non-linear similarity clustering** (t-SNE) isolate stable background forest baselines from post-event disturbed landscapes.

In [ ]:
print("1. Extracting high-dimensional structural representations from 'full_stack'...")
individual_embeddings = []

for i in range(12):
    single_image = full_stack[i] # Shape: (6, H, W)


    # Neutralize unpatched source NaNs for individual layers

    if np.isnan(single_image).any() or np.isinf(single_image).any():
        nan_count = np.isnan(single_image).sum()
        inf_count = np.isinf(single_image).sum()
        print(f"[FIX] Found pre-existing invalid numbers in Image Index [{i}] ({sorted_dates[i]})! NaNs: {nan_count}, Infs: {inf_count}. Patching...")

        # Replace bad values with a neutral 0.0 value
        single_image = np.nan_to_num(single_image, nan=0.0, posinf=0.0, neginf=0.0)

    # Apply HLS scaling calibration if it wasn't baked into full_stack yet
    if single_image.max() > 1.0:
        single_image = single_image * 0.0001

    # Structure tensor to match Prithvi format: [Batch=1, Channels=6, Time=4, Height, Width]
    # single_date_sequence adds the time dimension slot: (6, 1, H, W)
    single_date_sequence = np.expand_dims(single_image, axis=1)

    # Replicate along Time axis to satisfy Prithvi 4-timestep prerequisite: (6, 4, H, W)
    single_date_sequence = np.repeat(single_date_sequence, 4, axis=1)

    # Add batch dimension and move to device: [1, 6, 4, H, W]
    input_tensor = torch.from_numpy(single_date_sequence).float().unsqueeze(0).to(device)

    with torch.no_grad():
        try:
            features = model(input_tensor)
        except NameError:
            features = backbone(input_tensor)

        # Unpack layers safely based on your model's output configuration wrapper
        if isinstance(features, dict):
            feature_tensor = features.get("spatial_features", features.get("embeddings", list(features.values())[0]))
        elif isinstance(features, list):
            feature_tensor = features[-1]
        else:
            feature_tensor = features

        # Extract to CPU and flatten completely to preserve all spatial structural detail
        flat_vector = feature_tensor.detach().cpu().numpy().ravel()

        # Final inline fallback: Ensure the feature extraction itself didn't generate NaNs
        if np.isnan(flat_vector).any():
            flat_vector = np.nan_to_num(flat_vector, nan=0.0)

        individual_embeddings.append(flat_vector.tolist())

# Force stacking into a true 2D matrix layout
individual_embeddings_matrix = np.vstack([np.array(x) for x in individual_embeddings])
print(f"[SUCCESS]: Generated a true 2D matrix shape with ZERO NaNs: {individual_embeddings_matrix.shape}")


# 2. STANDARD SCALING

print("\n2. Normalizing individual date feature variances...")
scaler = preprocessing.StandardScaler()
scaled_matrix = scaler.fit_transform(individual_embeddings_matrix)

# Safety check for constant feature columns creating new scaling NaNs
if np.isnan(scaled_matrix).any():
    scaled_matrix = np.nan_to_num(scaled_matrix, nan=0.0)


# 3. LINEAR DIMENSIONALITY REDUCTION (PCA)

print("3. Projecting linear variance along orthogonal components...")
pca_individual = decomposition.PCA(n_components=2)
pca_ind_results = pca_individual.fit_transform(scaled_matrix)

var1 = pca_individual.explained_variance_ratio_[0] * 100
var2 = pca_individual.explained_variance_ratio_[1] * 100


# 4. NON-LINEAR DIMENSIONALITY REDUCTION (t-SNE)

print("4. Computing non-linear proximity manifolds via t-SNE...")
safe_perplexity = min(3, len(individual_embeddings_matrix) - 1)

tsne = TSNE(n_components=2, random_state=42, perplexity=safe_perplexity, n_iter=1000)
tsne_results = tsne.fit_transform(scaled_matrix)
print(" [SUCCESS]: Dimensionality reduction layers completed with clean values.")


# 5. INTEGRATED VISUALIZATION PIPELINE

fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(26, 7))
timeline_colors = plt.cm.coolwarm(np.linspace(0, 1, 12))

# --- Panel 1: PCA Component 1 ---
for i in range(12):
    ax1.scatter(i, pca_ind_results[i, 0], color=timeline_colors[i], s=250, edgecolors='black', zorder=3)
ax1.plot(range(12), pca_ind_results[:, 0], color='black', linestyle='-', alpha=0.4, linewidth=2, zorder=2)
ax1.axhline(0, color='gray', linestyle='--', linewidth=0.8)
ax1.set_xticks(range(12))
ax1.set_xticklabels(sorted_dates, rotation=45, ha='right')
ax1.set_xlabel("Exact Satellite Capture Date")
ax1.set_ylabel("PCA Component 1 Coordinate")
ax1.set_title(f"PCA Component 1 ({var1:.1f}%)")
ax1.grid(True, alpha=0.2)

# --- Panel 2: PCA Component 2 ---
for i in range(12):
    ax2.scatter(i, pca_ind_results[i, 1], color=timeline_colors[i], s=250, edgecolors='black', zorder=3)
ax2.plot(range(12), pca_ind_results[:, 1], color='black', linestyle='-', alpha=0.4, linewidth=2, zorder=2)
ax2.axhline(0, color='gray', linestyle='--', linewidth=0.8)
ax2.set_xticks(range(12))
ax2.set_xticklabels(sorted_dates, rotation=45, ha='right')
ax2.set_xlabel("Exact Satellite Capture Date")
ax2.set_ylabel("PCA Component 2 Coordinate")
ax2.set_title(f"PCA Component 2 ({var2:.1f}%)")
ax2.grid(True, alpha=0.2)

# --- Panel 3: t-SNE Non-Linear Neighborhood Groupings ---
for i in range(12):
    label_text = f"Day {i}: {sorted_dates[i]}"
    ax3.scatter(
        tsne_results[i, 0],
        tsne_results[i, 1],
        color=timeline_colors[i],
        s=300,
        edgecolors='black',
        label=label_text,
        zorder=3
    )
ax3.set_xlabel("t-SNE Dimension 1")
ax3.set_ylabel("t-SNE Dimension 2")
ax3.set_title("t-SNE Manifold Clusters (Semantic Proximity)")
ax3.grid(True, alpha=0.2)

plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', title="Exact Capture Timeline")
plt.tight_layout()
plt.show()

#Exercises

### Exercise 1: Generating Deep Geospatial Embeddings with Clay


Use **Clay** spatial foundation model designed for multi-spectral remote sensing analysis for the same analysis.

Unlike Prithvi (which uses a 3D spatio-temporal architecture requiring fixed sequence windows of 4 dates), Clay is built on a 2D Vision Transformer (ViT) architecture that handles standalone single-date observation captures natively. It processes individual spatial scenes into high-dimensional feature representations called **embeddings** that preserve fine-grained localized land disturbances, vegetation shifts, and canopy state conditions without mixing temporal inputs.

Initialize pre-trained weights of **Clay v1.5** via the `terratorch` registry (`timm_clay_v1_base`)

**what changes?**

### Step 10: Data Preparation & Normalization

Before feeding the images into Clay, we prepare the raw GEE data into model-ready tensors in three steps:


1. **Normalize with HLS statistics** — each band is normalized using HLS-specific mean and standard deviation values (pre-computed from global HLS data), ensuring the input distribution matches what Clay was trained on

2. **Crop & reformat each scene:**
   - Transposed from HWC → CHW format as expected by PyTorch
   - Stacked into a single array of shape **(12, 6, 256, 256)** — 12 dates × 6 bands × 256 × 256 pixels

## Exercise 2: Unsupervised Clustering on Embeddings

Can the embeddings naturally separate pre- and post-Vaia scenes without using labels?

### How to do it
- Use the extracted embedding matrix (`12 × D`)
- Apply PCA to reduce the embeddings to two dimensions
- Use `KMeans(n_clusters=2)` to group the scenes
- Visualize the clusters and compare them with the acquisition dates

### What to expect
- A clear separation between pre- and post-Vaia scenes suggests the embeddings capture storm-related changes
- Mixed clusters may indicate that seasonal effects, noise, or scene selection have a stronger influence than the storm signal

> 💡 Any separation emerges directly from the structure of the embedding space.

### Exercise 2.1: Random Forest Classification on Embeddings

Can we **classify** each scene as pre- or post-Vaia using the embeddings as input features?

**How to do it:**
- Use `full_stack` (shape `12 × 6 × 256 × 256`) — flatten each scene's embedding into a 1D feature vector
- Train a `RandomForestClassifier` from scikit-learn using a leave-one-out or train/test split

**What to expect:**
- High accuracy (ideally 100%) would confirm the embeddings carry enough discriminative information to separate pre- from post-storm scenes
- Low accuracy suggests the selected scenes are too noisy or too few — a good reason to go back and refine your scene selection



### Exercise 3: What is the effect of Noisy Scenes?

- Snowy scenes formed their **own cluster**, since snow has a very distinct spectral signature (high reflectance across visible bands, low in SWIR)

- Re-run the pipeline excluding **cloudy or snowy images** in Step 6 and observe how the embedding space reacts.

**How to do it:**
- Normalize or remove the snowy/cloudy scene indices you noted during visual inspection
- Re-run later steps with the new image set

**what changes?**




